## ***Horizon League***

***Schools***
1. Cleveland State Vikings
2. Detroit Mercy Titans
3. UW Green Bay Phoenix
4. IU Indianapolis Jaguars
5. UW Milwaukee Panthers
6. Northern Kentucky Norse
7. Oakland Golden Grizzlies
8. Purdue Fort Wayne Mastodons
9. Robert Morris Colonials
10. Wright State Raiders
11. Youngstown State Penguins

In [2]:
### Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

***Scrapers***

In [ ]:
"""
Sidearm Sports <tr> Scraper for Horizon League

Teams:
- Most of Atlantic Sun, Atlantic 10
- Big East: Butler, Georgetown, Villanova
- Big Sky: UC Davis, Eastern Washington, Idaho State, Idaho, Northern Arizona, Portland State, Sacramento State
- Big South: Charleston Southern, Gardner-Webb, High Point, Longwood, Presbyterian, Radford, USC Upstate, Winthrop
- Big West: Cal State Bakersfield, Cal State Fullerton, Hawai'i, UC Irvine, UC Riverside, UC San Diego, UC Santa Barbara
- Big 10: Illinois, Maryland, Rutgers
- CAA: Campbell, Elon, Hampton, Hofstra, Monmouth, North Carolina A&T, Northeastern, Stony Brook, Towson, UNC Wilmington, William & Mary
- Horizon League: Cleveland State, Detroit Mercy, Green Bay, IU Indianapolis, Milwaukee, Northern Kentucky, Oakland, Robert Morris, Wright State, Youngstown State
"""
def scrape_sidearm_tr(url, school, conference = "Horizon League"):
    
    # Set custom headers to mimic a browser request
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    # Get url, parse
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, "html.parser")

    # Initialize variables
    staff_list = []
    current_department = None

    # Iterate through table rows
    for row in soup.find_all("tr"):
        row_class = [cls.strip() for cls in row.get("class", [])]

        # Department header
        if any("sidearm-staff-category" in c for c in row_class):
            dept_cell = row.find("td", class_="fake-heading")
            if dept_cell:
                current_department = dept_cell.get_text(strip=True)

        # Staff member row
        elif any("sidearm-staff-member" in c for c in row_class):
            name_tag = row.find("a", href=True)
            name = name_tag.get_text(strip=True) if name_tag else None

            title_cell = row.find("td", headers=lambda h: h and "col-staff_title" in h)
            if not title_cell:
                all_tds = row.find_all("td")
                title_cell = all_tds[2] if len(all_tds) > 2 else None
            title = title_cell.get_text(strip=True) if title_cell else None

            if name:
                staff_list.append({
                    "name": name,
                    "title": title or "",
                    "department": current_department or "",
                    "conference": conference,
                    "school": school
                })

    df = pd.DataFrame(staff_list)
    print(f"{school}: Scraped {len(df)} Staff Members")
    return df


In [12]:
'''
<div>, <h3>, <h4> BeautifulSoup Scraper
Teams:
- North Florida (Atlantic Sun)
- Davidson (Atlantic 10)
- Saint Louis (Atlantic 10)
- Big East: Creighton, UConn, DePaul, Marquette
- Big Sky: Montana State, Northern Colorado
- Big South: UNC Asheville
- Big West: Cal Poly, CSUN, Long Beach State
- Big 10: Indiana, Michigan, Michigan State, Minnesota, Northwestern, Ohio State, Washington
- Big 12: Baylor, Iowa State, Kansas, Kansas State, Oklahoma State, TCU, Utah, West Virginia
- CAA: Charleston, Drexel
- Horizon League: Purdue Fort Wayne
'''
def scrape_bs4_div_h3_h4(url, school, conference = "Coastal Athletic Association"):
    # Make sure school can be fetched
    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    # Set up parser, variables
    soup = BeautifulSoup(response.text, "html.parser")
    staff_list = []
    current_department = None

    # Iterate through divs and h3s to find departments and staff cards
    for element in soup.find_all(["div", "h3"]):
        # If it's a department header
        if element.name == "h3" and "s-text-title" in element.get("class", []):
            current_department = element.get_text(strip=True)

        # If it's a staff card
        elif element.get("data-test-id") == "s-person-card-list__root":
            name_tag = element.find("h4")
            name = name_tag.get_text(strip=True) if name_tag else None

            title_tag = element.find("div", class_="s-person-details__position")
            title = title_tag.get_text(" ", strip=True) if title_tag else None

            if name:  # Skip blank cards
                staff_list.append({
                    "name": name,
                    "title": title,
                    "department": current_department,
                    "school": school,
                    "conference": conference
                })

    df = pd.DataFrame(staff_list)
    print(f"{school}: Scraped {len(df)} staff members.")
    time.sleep(2)  # gentle delay
    return df

***Individual School Checks***

In [ ]:
### Cleveland State
school_df = scrape_sidearm_tr(url = "https://csuvikings.com/staff-directory", school = "Cleveland State")
school_df['department'].value_counts()

In [ ]:
### Detroit Mercy
school_df = scrape_sidearm_tr(url = "https://detroittitans.com/staff-directory", school = "Detroit Mercy")
school_df['department'].value_counts()

In [ ]:
### Green Bay
school_df = scrape_sidearm_tr(url = "https://greenbayphoenix.com/staff-directory", school = "Green Bay")
school_df['department'].value_counts()

In [ ]:
## IU Indy
iu_indy_df = scrape_sidearm_tr(url = "https://iuindyjags.com/staff-directory", school = "IU Indianapolis")
iu_indy_df['department'].value_counts()

In [ ]:
### Milwaukee
school_df = scrape_sidearm_tr(url = "https://mkepanthers.com/staff-directory", school = "Milwaukee")
school_df['department'].value_counts()

In [ ]:
### Northern Kentucky
school_df = scrape_sidearm_tr(url = "https://nkunorse.com/staff-directory", school = "Northern Kentucky")
school_df['department'].value_counts()

In [ ]:
### Oakland
school_df = scrape_sidearm_tr(url = "https://goldengrizzlies.com/staff-directory", school = "Oakland")
school_df['department'].value_counts()

In [ ]:
### Purdue Fort Wayne
school_df = scrape_bs4_div_h3_h4(url = "https://gomastodons.com/staff-directory", school = "Purdue Fort Wayne")
school_df['department'].value_counts()

In [ ]:
### Robert Morris
school_df = scrape_sidearm_tr(url = "https://rmucolonials.com/staff-directory", school = "Robert Morris")
school_df['department'].value_counts()

In [ ]:
### Wright State
school_df = scrape_sidearm_tr(url = "https://wsuraiders.com/staff-directory", school = "Wright State")
school_df['department'].value_counts()

In [ ]:
### Youngstown State
school_df = scrape_sidearm_tr(url = "https://ysusports.com/staff-directory", school = "Youngstown State")
school_df['department'].value_counts()

***Scrape Horizon League***

In [18]:
def scrape_horizon():
    schools = [
        {'school': 'Cleveland State', 'url': 'https://csuvikings.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Detroit Mercy', 'url': 'https://detroittitans.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Green Bay', 'url': 'https://greenbayphoenix.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'IU Indianapolis', 'url': 'https://iuindyjags.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Milwaukee', 'url': 'https://mkepanthers.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Northern Kentucky', 'url': 'https://nkunorse.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Oakland', 'url': 'https://goldengrizzlies.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Purdue Fort Wayne', 'url': 'https://gomastodons.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'Horizon League'},
        {'school': 'Robert Morris', 'url': 'https://rmucolonials.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Wright State', 'url': 'https://wsuraiders.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
        {'school': 'Youngstown State', 'url': 'https://ysusports.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Horizon League'},
    ]
    
    all_staff = []
    for s in schools:
        if s['scraper'] == 'sidearm':
            df = scrape_sidearm_tr(url = s['url'], school = s['school'], conference = s['conference'])
        elif s['scraper'] == 'div_h3_h4':
            df = scrape_bs4_div_h3_h4(url = s['url'], school = s['school'], conference = s['conference'])
            
        all_staff.append(df)
        
    combined_df = pd.concat(all_staff, ignore_index=True)
    return combined_df

In [ ]:
hl_df = scrape_horizon()
hl_df.to_csv("horizon_league_staff_directory.csv", index=False)